# ClinicalBERT Fine-Tuning

Stages:
- DAPT (masked language modeling) on unlabeled text
- Supervised fine-tuning for disease classification

Configuration is read from `.env` (paths, hyperparameters, and run flags).

In [ ]:
# Dependencies (optional)
!pip install -q transformers datasets evaluate scikit-learn accelerate python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.7 MB/s eta 0:00:00


In [ ]:
# Colab Drive mount (optional)
try:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except Exception:
    pass

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import evaluate
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

# --- Optional .env loading ----------------------------------------------------
try:
    from dotenv import load_dotenv  # type: ignore

    load_dotenv(override=False)
except Exception:
    print("python-dotenv not installed (or .env missing); continuing without it.")


def _env_bool(name: str, default: str = "0") -> bool:
    return os.getenv(name, default).strip().lower() in {"1", "true", "yes", "y"}


SEED = int(os.getenv("SEED", "27"))
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device} | seed={SEED}")

RUN_DAPT = _env_bool("RUN_DAPT", "0")
RUN_FINETUNE = _env_bool("RUN_FINETUNE", "0")
RUN_EVAL = _env_bool("RUN_EVAL", "0")

Using device: cuda


In [ ]:
def _env_str(name: str, default: str) -> str:
    return os.getenv(name, default).strip()


# ---------------------------- Base model -------------------------------------
BASE_MODEL_NAME = _env_str("BASE_MODEL_NAME", "emilyalsentzer/Bio_ClinicalBERT")

# ---------------------------- Data paths -------------------------------------
# Defaults are Colab-style paths; override via `.env` when running locally.
DAPT_DATASET1_PATH = _env_str("DAPT_DATASET1_PATH", "/content/drive/MyDrive/Datasets/dataset1.csv")
DAPT_DATASET2_PATH = _env_str("DAPT_DATASET2_PATH", "/content/drive/MyDrive/Datasets/dataset2.csv")
DAPT_TEXT_COLUMN = _env_str("DAPT_TEXT_COLUMN", "text")

CLASSIFIER_DATASET_PATH = _env_str(
    "CLASSIFIER_DATASET_PATH",
    "/content/drive/MyDrive/Datasets/Final_Augmented_dataset_Diseases_and_Symptoms.csv",
)

DAPT_OUTPUT_DIR = _env_str("DAPT_OUTPUT_DIR", "./dapt_model")
DAPT_ADAPTED_DIR = _env_str("DAPT_ADAPTED_DIR", "./dapt_adapted_model")
FINETUNE_OUTPUT_DIR = _env_str("FINETUNE_OUTPUT_DIR", "./hf_disease_classifier")
FINAL_MODEL_DIR = _env_str(
    "FINAL_MODEL_DIR",
    "/content/drive/MyDrive/fine tuned model & tokenizer/final_model-2",
)

DAPT_NUM_EPOCHS = int(_env_str("DAPT_NUM_EPOCHS", "3"))
DAPT_BATCH_SIZE = int(_env_str("DAPT_BATCH_SIZE", "16"))
DAPT_LEARNING_RATE = float(_env_str("DAPT_LEARNING_RATE", "5e-5"))
DAPT_WEIGHT_DECAY = float(_env_str("DAPT_WEIGHT_DECAY", "0.01"))
DAPT_WARMUP_STEPS = int(_env_str("DAPT_WARMUP_STEPS", "500"))
DAPT_SAVE_STEPS = int(_env_str("DAPT_SAVE_STEPS", "500"))
DAPT_SAVE_TOTAL_LIMIT = int(_env_str("DAPT_SAVE_TOTAL_LIMIT", "2"))
DAPT_LOGGING_STEPS = int(_env_str("DAPT_LOGGING_STEPS", "100"))
DAPT_MLM_PROB = float(_env_str("DAPT_MLM_PROB", "0.15"))
DAPT_MAX_LENGTH = int(_env_str("DAPT_MAX_LENGTH", "512"))

CLS_MAX_LENGTH = int(_env_str("CLS_MAX_LENGTH", "256"))
CLS_NUM_EPOCHS = int(_env_str("CLS_NUM_EPOCHS", "4"))
CLS_BATCH_SIZE = int(_env_str("CLS_BATCH_SIZE", "16"))
CLS_LEARNING_RATE = float(_env_str("CLS_LEARNING_RATE", "2e-5"))
TEST_SIZE = float(_env_str("TEST_SIZE", "0.10"))
VALID_SIZE = float(_env_str("VALID_SIZE", "0.30"))
RANDOMIZE_SYMPTOM_ORDER = _env_bool("RANDOMIZE_SYMPTOM_ORDER", "1")

# Requested backbone for the classifier. We resolve the fallback *later*, at load-time,
# so a "run DAPT -> save -> fine-tune" workflow works without re-running config cells.
CLS_MODEL_NAME = _env_str("CLS_MODEL_NAME", DAPT_ADAPTED_DIR)

print("Config summary:")
print(f"- BASE_MODEL_NAME: {BASE_MODEL_NAME}")
print(f"- CLS_MODEL_NAME (requested): {CLS_MODEL_NAME}")
print(f"- DAPT_TEXT_COLUMN: {DAPT_TEXT_COLUMN}")
print(f"- Output dirs: DAPT_ADAPTED_DIR='{DAPT_ADAPTED_DIR}', FINETUNE_OUTPUT_DIR='{FINETUNE_OUTPUT_DIR}'")

print("\nPath checks (non-fatal):")
for k, v in {
    "DAPT_DATASET1_PATH": DAPT_DATASET1_PATH,
    "DAPT_DATASET2_PATH": DAPT_DATASET2_PATH,
    "CLASSIFIER_DATASET_PATH": CLASSIFIER_DATASET_PATH,
    "CLS_MODEL_NAME": CLS_MODEL_NAME,
}.items():
    exists = Path(v).exists()
    print(f"- {k}: {'OK' if exists else 'MISSING'} -> {v}")

Dataset 1 shape: (1409, 1)
Dataset 2 shape: (5634, 1)

 Combined DAPT dataset shape: (7043, 1)
Total samples for DAPT: 7043

Sample texts:
                                                text
0  itching,vomiting,fatigue,weight_loss,high_feve...
1  fatigue,weight_loss,restlessness,sweating,diar...
2  constipation,pain_during_bowel_movements,pain_...
3  chills,vomiting,fatigue,weight_loss,cough,high...
4  vomiting,yellowish_skin,abdominal_pain,swellin...


## Stage 1 — DAPT

Masked LM training on domain text (controlled by `RUN_DAPT`).

In [ ]:
print(f"Loading base model/tokenizer: {BASE_MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
mlm_model = AutoModelForMaskedLM.from_pretrained(BASE_MODEL_NAME)

print("Base MLM model loaded successfully")

Loading pre-trained model: emilyalsentzer/Bio_ClinicalBERT


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

 Pre-trained model loaded successfully


In [ ]:
def load_dapt_dataframe() -> pd.DataFrame:
    df1 = pd.read_csv(DAPT_DATASET1_PATH)
    df2 = pd.read_csv(DAPT_DATASET2_PATH)

    dapt_df = pd.concat([df1, df2], ignore_index=True)

    if DAPT_TEXT_COLUMN not in dapt_df.columns:
        raise ValueError(
            f"Expected text column '{DAPT_TEXT_COLUMN}' not found. "
            f"Available columns: {list(dapt_df.columns)}"
        )

    return dapt_df[[DAPT_TEXT_COLUMN]].rename(columns={DAPT_TEXT_COLUMN: "text"})


def tokenize_dapt_dataset(dapt_df: pd.DataFrame) -> Dataset:
    dapt_dataset = Dataset.from_pandas(dapt_df)

    def tokenize_function(examples):
        return tokenizer(
            examples["text"],
            truncation=True,
            padding=False,
            max_length=DAPT_MAX_LENGTH,
        )

    return dapt_dataset.map(
        tokenize_function,
        batched=True,
        remove_columns=dapt_dataset.column_names,
    )


tokenized_dapt = None

dapt_data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=DAPT_MLM_PROB,
)

if RUN_DAPT:
    dapt_df = load_dapt_dataframe()
    tokenized_dapt = tokenize_dapt_dataset(dapt_df)
else:
    pass

Tokenizing DAPT dataset...


Map:   0%|          | 0/7043 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

 Tokenization complete


In [ ]:
if RUN_DAPT:
    if tokenized_dapt is None:
        raise RuntimeError("tokenized_dapt is None. Run the DAPT preparation cell first.")

    dapt_training_args = TrainingArguments(
        output_dir=DAPT_OUTPUT_DIR,
        overwrite_output_dir=True,
        num_train_epochs=DAPT_NUM_EPOCHS,
        per_device_train_batch_size=DAPT_BATCH_SIZE,
        save_steps=DAPT_SAVE_STEPS,
        save_total_limit=DAPT_SAVE_TOTAL_LIMIT,
        logging_steps=DAPT_LOGGING_STEPS,
        learning_rate=DAPT_LEARNING_RATE,
        weight_decay=DAPT_WEIGHT_DECAY,
        warmup_steps=DAPT_WARMUP_STEPS,
        fp16=True if device == "cuda" else False,
        report_to="none",
    )

    dapt_trainer = Trainer(
        model=mlm_model,
        args=dapt_training_args,
        train_dataset=tokenized_dapt,
        data_collator=dapt_data_collator,
    )

    print("Starting DAPT training...")
    print(f"Training on {len(tokenized_dapt)} samples")
    dapt_trainer.train()
    print("DAPT training completed.")
else:
    print("RUN_DAPT is disabled; skipping DAPT training.")

Starting DAPT phase...
Training on 7043 samples


Step,Training Loss
100,1.835800
200,1.452000
300,1.211200
400,0.975100
500,0.919000
600,0.859900
700,0.838100
800,0.766500
900,0.762200
1000,0.734400



 DAPT phase completed!


In [ ]:
if RUN_DAPT:
    Path(DAPT_ADAPTED_DIR).mkdir(parents=True, exist_ok=True)
    mlm_model.save_pretrained(DAPT_ADAPTED_DIR)
    tokenizer.save_pretrained(DAPT_ADAPTED_DIR)
    print(f"DAPT-adapted model saved to: {DAPT_ADAPTED_DIR}")
else:
    print("RUN_DAPT is disabled; skipping DAPT model saving.")
    print(f"(Expected DAPT_ADAPTED_DIR: {DAPT_ADAPTED_DIR})")

 DAPT model saved to ./dapt_adapted_model


## Stage 2 — Supervised Fine-Tuning

Sequence classification training + evaluation (controlled by `RUN_FINETUNE` / `RUN_EVAL`).

In [ ]:
def build_processed_classifier_df(csv_path: str) -> pd.DataFrame:
    """Load the symptom dataset and convert each row into a comma-separated symptom text."""
    df = pd.read_csv(csv_path)

    # Identify columns (assumes label is the first column)
    label_col = df.columns[0]
    symptom_cols = df.columns[1:]

    def row_to_symptoms(row: pd.Series) -> str:
        symptoms = [symptom for symptom, val in row.items() if val == 1]
        if RANDOMIZE_SYMPTOM_ORDER:
            random.shuffle(symptoms)
        return ", ".join(symptoms)

    df["symptom_text"] = df[symptom_cols].apply(row_to_symptoms, axis=1)

    processed = (
        df[[label_col, "symptom_text"]]
        .rename(columns={label_col: "disease"})
        .sample(frac=1, random_state=SEED)
        .reset_index(drop=True)
    )
    return processed


processed_df = None
if Path(CLASSIFIER_DATASET_PATH).exists():
    processed_df = build_processed_classifier_df(CLASSIFIER_DATASET_PATH)
    print("Sample data after preprocessing:")
    print(processed_df.head())
else:
    print("Classifier dataset not found; skipping preprocessing.")
    print(f"Expected CLASSIFIER_DATASET_PATH: {CLASSIFIER_DATASET_PATH}")

 Sample data after preprocessing:
                   disease                                       symptom_text
0         chronic glaucoma  itchiness of eye, spots or clouds in vision, p...
1              torticollis  headache, throat swelling, abnormal involuntar...
2     personality disorder  insomnia, temper problems, excessive anger, lo...
3  fracture of the patella  hip pain, knee swelling, leg pain, leg weaknes...
4     chronic otitis media  redness in ear, ear pain, allergic reaction, f...


In [ ]:
if processed_df is None:
    print("processed_df is None (dataset missing).")
else:
    processed_df.head()

,disease,symptom_text
0,chronic glaucoma,"itchiness of eye, spots or clouds in vision, p..."
1,torticollis,"headache, throat swelling, abnormal involuntar..."
2,personality disorder,"insomnia, temper problems, excessive anger, lo..."
3,fracture of the patella,"hip pain, knee swelling, leg pain, leg weaknes..."
4,chronic otitis media,"redness in ear, ear pain, allergic reaction, f..."


In [ ]:
hf_dataset = None

if processed_df is None:
    print("Skipping train/val/test split because processed_df is None.")
else:
    # Split sizes
    # - TEST_SIZE is taken from the full dataset
    # - VALID_SIZE is taken from the remaining (train+val) portion
    try:
        train_val_df, test_df = train_test_split(
            processed_df,
            test_size=TEST_SIZE,
            random_state=SEED,
            stratify=processed_df["disease"],
        )
    except ValueError as exc:
        print(f"Stratified test split failed: {exc}")
        train_val_df, test_df = train_test_split(
            processed_df,
            test_size=TEST_SIZE,
            random_state=SEED,
            stratify=None,
        )

    try:
        train_df, valid_df = train_test_split(
            train_val_df,
            test_size=VALID_SIZE,
            random_state=SEED,
            stratify=train_val_df["disease"],
        )
    except ValueError as exc:
        print(f"Stratified validation split failed: {exc}")
        train_df, valid_df = train_test_split(
            train_val_df,
            test_size=VALID_SIZE,
            random_state=SEED,
            stratify=None,
        )

    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)
    test_df = test_df.reset_index(drop=True)

    print(f"Splits: train={len(train_df)}, val={len(valid_df)}, test={len(test_df)}")

    hf_dataset = DatasetDict(
        {
            "train": Dataset.from_pandas(train_df),
            "validation": Dataset.from_pandas(valid_df),
            "test": Dataset.from_pandas(test_df),
        }
    )

Splits: train=155576, val=66675, test=24694


In [ ]:
label_encoder = None
id2label = None
label2id = None
num_labels = None

if hf_dataset is None or processed_df is None:
    print("Skipping label encoding because the dataset is missing.")
else:
    label_encoder = LabelEncoder()
    all_labels = processed_df["disease"].astype(str).unique().tolist()
    label_encoder.fit(all_labels)

    def encode_labels(example):
        example["labels"] = int(label_encoder.transform([str(example["disease"])])[0])
        return example

    hf_dataset = hf_dataset.map(encode_labels)

    id2label = {int(i): label for i, label in enumerate(label_encoder.classes_)}
    label2id = {label: int(i) for i, label in enumerate(label_encoder.classes_)}

    num_labels = len(label_encoder.classes_)
    print("Number of disease classes:", num_labels)
    print("Some label => id examples:", dict(list(label2id.items())[:10]))

Map:   0%|          | 0/155576 [00:00<?, ? examples/s]

Map:   0%|          | 0/66675 [00:00<?, ? examples/s]

Map:   0%|          | 0/24694 [00:00<?, ? examples/s]

Number of disease classes: 773
Some label => id examples: {np.str_('abdominal aortic aneurysm'): 0, np.str_('abdominal hernia'): 1, np.str_('abscess of nose'): 2, np.str_('abscess of the lung'): 3, np.str_('abscess of the pharynx'): 4, np.str_('acanthosis nigricans'): 5, np.str_('acariasis'): 6, np.str_('achalasia'): 7, np.str_('acne'): 8, np.str_('actinic keratosis'): 9}


In [ ]:
tokenized_dataset = None
cls_tokenizer = None
cls_model = None

if hf_dataset is None or num_labels is None or id2label is None or label2id is None:
    print("Skipping tokenization/model init because dataset/labels are missing.")
else:
    backbone = CLS_MODEL_NAME
    if Path(backbone).exists():
        print(f"Loading classifier backbone from local path: {backbone}")
    else:
        print(f"Classifier backbone path not found: {backbone}")
        print(f"Falling back to BASE_MODEL_NAME: {BASE_MODEL_NAME}")
        backbone = BASE_MODEL_NAME

    cls_tokenizer = AutoTokenizer.from_pretrained(backbone, use_fast=True)

    def tokenize_fn(batch):
        return cls_tokenizer(
            batch["symptom_text"],
            truncation=True,
            max_length=CLS_MAX_LENGTH,
        )

    tokenized_dataset = hf_dataset.map(tokenize_fn, batched=True)

    cls_model = AutoModelForSequenceClassification.from_pretrained(
        backbone,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    )
    cls_model.to(device)
    print("Classifier model initialized.")

Map:   0%|          | 0/155576 [00:00<?, ? examples/s]

Map:   0%|          | 0/66675 [00:00<?, ? examples/s]

Map:   0%|          | 0/24694 [00:00<?, ? examples/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at ./dapt_adapted_model and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
_accuracy_metric = None
_f1_metric = None


def compute_metrics(eval_pred):
    """Compute accuracy + macro-F1 (robust to class imbalance)."""
    global _accuracy_metric, _f1_metric

    if _accuracy_metric is None:
        _accuracy_metric = evaluate.load("accuracy")
    if _f1_metric is None:
        _f1_metric = evaluate.load("f1")

    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = _accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    f1_macro = _f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1_macro}


print("compute_metrics() ready: accuracy + macro-F1")

In [ ]:
trainer = None

if tokenized_dataset is None or cls_model is None or cls_tokenizer is None:
    print("Skipping Trainer setup because prerequisites are missing.")
else:
    data_collator = DataCollatorWithPadding(tokenizer=cls_tokenizer)

    training_args = TrainingArguments(
        output_dir=FINETUNE_OUTPUT_DIR,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",
        learning_rate=CLS_LEARNING_RATE,
        per_device_train_batch_size=CLS_BATCH_SIZE,
        per_device_eval_batch_size=CLS_BATCH_SIZE,
        num_train_epochs=CLS_NUM_EPOCHS,
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        seed=SEED,
        fp16=True if device == "cuda" else False,
        report_to="none",
    )

    trainer = Trainer(
        model=cls_model,
        args=training_args,
        train_dataset=tokenized_dataset["train"],
        eval_dataset=tokenized_dataset["validation"],
        tokenizer=cls_tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    print("Trainer initialized.")

/tmp/ipython-input-2328976188.py:26: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
if not RUN_FINETUNE:
    print("RUN_FINETUNE is disabled; skipping supervised fine-tuning.")
elif trainer is None:
    raise RuntimeError("Trainer is None. Run the Trainer setup cell first.")
else:
    trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro
1,0.490100,0.459329,0.849164,0.728933
2,0.403300,0.423444,0.855883,0.766252
3,0.342800,0.406650,0.857458,0.776456


TrainOutput(global_step=29172, training_loss=0.41207933863968443, metrics={'train_runtime': 2585.3793, 'train_samples_per_second': 180.526, 'train_steps_per_second': 11.283, 'total_flos': 9616170022783296.0, 'train_loss': 0.41207933863968443, 'epoch': 3.0})

In [ ]:
if not RUN_EVAL:
    print("RUN_EVAL is disabled; skipping evaluation + saving.")
elif trainer is None or tokenized_dataset is None or cls_tokenizer is None:
    raise RuntimeError("Missing trainer/tokenized_dataset/tokenizer. Run the previous cells first.")
else:
    metrics = trainer.evaluate(tokenized_dataset["test"])
    print("Test metrics:")

    metrics_df = pd.DataFrame([metrics])
    try:
        from IPython.display import display  # type: ignore

        display(metrics_df)
    except Exception:
        print(metrics_df)

    output_dir = Path(FINAL_MODEL_DIR)
    output_dir.mkdir(parents=True, exist_ok=True)

    trainer.save_model(str(output_dir))
    cls_tokenizer.save_pretrained(str(output_dir))

    # Save label mappings for easier downstream inference
    if label2id is not None and id2label is not None:
        import json

        (output_dir / "label2id.json").write_text(
            json.dumps(label2id, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        (output_dir / "id2label.json").write_text(
            json.dumps(id2label, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )

    print(f"Model + tokenizer saved to: {output_dir}")

Test metrics:


,eval_loss,eval_accuracy,eval_f1_macro,eval_runtime,eval_samples_per_second,eval_steps_per_second,epoch
0,0.40246,0.859197,0.798047,26.7303,923.822,57.762,3.0


Fine-tuned model and tokenizer saved to /content/drive/MyDrive/fine tuned model & tokenizer/final_model-2
